In [1]:
#depth calo
X0_ECAL = 3.897
X0_HCAL = 2.357 

Lambda_int_ECAL = 37.578
Lambda_int_HCAL = 17.438

depth=[ 3*X0_ECAL/Lambda_int_ECAL,16*X0_ECAL/Lambda_int_ECAL,6*X0_ECAL/Lambda_int_ECAL, 1.5, 4.1,1.8 ]
print(depth)

[0.3111128851987865, 1.6592687210601946, 0.622225770397573, 1.5, 4.1, 1.8]


In [2]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import ROOT
import random

Welcome to JupyROOT 6.16/00


In [3]:
#file wo noise
f = h5py.File('../Outfile_CellInformation_HomDet_2to5GeV.h5','r')

#charged+noise
f_ch= h5py.File('../Outfile_CellInformation_NoisyCharged.h5','r')

In [48]:
f.keys()

<KeysViewHDF5 ['MidPoint_X_Layer1', 'MidPoint_X_Layer2', 'MidPoint_X_Layer3', 'MidPoint_X_Layer4', 'MidPoint_X_Layer5', 'MidPoint_X_Layer6', 'MidPoint_Y_Layer1', 'MidPoint_Y_Layer2', 'MidPoint_Y_Layer3', 'MidPoint_Y_Layer4', 'MidPoint_Y_Layer5', 'MidPoint_Y_Layer6', 'MidPoint_Z_Layer', 'Photon1_E', 'Photon1_Phi', 'Photon1_Theta', 'Photon2_E', 'Photon2_Phi', 'Photon2_Theta', 'Pi0_Phi', 'Pi0_Theta', 'RealRes_ChargedEnergy_Layer1', 'RealRes_ChargedEnergy_Layer2', 'RealRes_ChargedEnergy_Layer3', 'RealRes_ChargedEnergy_Layer4', 'RealRes_ChargedEnergy_Layer5', 'RealRes_ChargedEnergy_Layer6', 'RealRes_NeutralEnergy_Layer1', 'RealRes_NeutralEnergy_Layer2', 'RealRes_NeutralEnergy_Layer3', 'RealRes_NeutralEnergy_Layer4', 'RealRes_NeutralEnergy_Layer5', 'RealRes_NeutralEnergy_Layer6', 'RealRes_TotalEnergy_Layer1', 'RealRes_TotalEnergy_Layer2', 'RealRes_TotalEnergy_Layer3', 'RealRes_TotalEnergy_Layer4', 'RealRes_TotalEnergy_Layer5', 'RealRes_TotalEnergy_Layer6', 'Smeared_Track_Energy', 'Track_Imag

In [4]:
#cluster on total
Cluster_Total  = h5py.File('../DATA_PROCESSING/Outfile_TopoCluster_Total_2N.h5','r')

#cluster on charge wo noise
Cluster_Ch= h5py.File('../DATA_PROCESSING/Outfile_TopoCluster_Charged_2N.h5','r')

#cluster on charged+noise
Cluster_Charged = h5py.File('../DATA_PROCESSING/Outfile_TopoCluster_ChargedNoisy_2N.h5','r')

In [5]:
#calorimeter struct
import math

def MakeTruthTrajectory(Theta, Phi, x_indx, y_indx) : 


    X0_ECAL = 3.897
    Lambda_int_HCAL = 17.438

    Total_ECAL_Length = 3 * X0_ECAL + 16 * X0_ECAL + 6 * X0_ECAL
    Total_HCAL_Length = 1.5 * Lambda_int_HCAL + 4.1 * Lambda_int_HCAL + 1.8 * Lambda_int_HCAL
    Total_Calo_Length = Total_ECAL_Length + Total_HCAL_Length + 1.0 # -- there is a 1 cm gap between ECAL & HCAL


    zpos_ECAL1 = -1 * Total_Calo_Length/2 + 3 * X0_ECAL/2
    zpos_ECAL2 = zpos_ECAL1 + 3 * X0_ECAL/2 + 16  * X0_ECAL/2
    zpos_ECAL3 = zpos_ECAL2 + 16 * X0_ECAL/2 + 6 * X0_ECAL/2

    zpos_GAP = zpos_ECAL3 + 6 * X0_ECAL/2 + 1/2

    zpos_HCAL1 = zpos_GAP + 1/2 + 1.5 * Lambda_int_HCAL/2
    zpos_HCAL2 = zpos_HCAL1 + 1.5 * Lambda_int_HCAL/2 + 4.1 * Lambda_int_HCAL/2
    zpos_HCAL3 = zpos_HCAL2 + 4.1 * Lambda_int_HCAL/2 + 1.8 * Lambda_int_HCAL/2

    layer_struct = {'layer1' : 32, 'layer2' : 64, 'layer3' : 32, 'layer4' : 16, 'layer5' : 16, 'layer6' : 8}


    x_orig, y_orig = 0., 0.
    z_orig = -1 * (Total_Calo_Length/2 + 250)

    if(x_indx == 5 and y_indx == 5) : x_orig, y_orig   =  20.,  20.
    if(x_indx == 5 and y_indx == -5) : x_orig, y_orig  =  20., -20.
    if(x_indx == -5 and y_indx == 5) : x_orig, y_orig  = -20.,  20.
    if(x_indx == -5 and y_indx == -5) : x_orig, y_orig = -20., -20.

    layer1_x = x_orig + (zpos_ECAL1 - z_orig) * math.tan(Theta) * math.cos(Phi)
    layer1_y = y_orig + (zpos_ECAL1 - z_orig) * math.tan(Theta) * math.sin(Phi)

    layer2_x = x_orig + (zpos_ECAL2 - z_orig) * math.tan(Theta) * math.cos(Phi)
    layer2_y = y_orig + (zpos_ECAL2 - z_orig) * math.tan(Theta) * math.sin(Phi)

    layer3_x = x_orig + (zpos_ECAL3 - z_orig) * math.tan(Theta) * math.cos(Phi)
    layer3_y = y_orig + (zpos_ECAL3 - z_orig) * math.tan(Theta) * math.sin(Phi)

    layer4_x = x_orig + (zpos_HCAL1 - z_orig) * math.tan(Theta) * math.cos(Phi)
    layer4_y = y_orig + (zpos_HCAL1 - z_orig) * math.tan(Theta) * math.sin(Phi)

    layer5_x = x_orig + (zpos_HCAL2 - z_orig) * math.tan(Theta) * math.cos(Phi)
    layer5_y = y_orig + (zpos_HCAL2 - z_orig) * math.tan(Theta) * math.sin(Phi)

    layer6_x = x_orig + (zpos_HCAL3 - z_orig) * math.tan(Theta) * math.cos(Phi)
    layer6_y = y_orig + (zpos_HCAL3 - z_orig) * math.tan(Theta) * math.sin(Phi)

    CellIndexLayer = []

    CellIndexLayer.append(  FindCellIndex(layer1_x, layer1_y, 'layer1') )
    CellIndexLayer.append(  FindCellIndex(layer2_x, layer2_y, 'layer2') )
    CellIndexLayer.append(  FindCellIndex(layer3_x, layer3_y, 'layer3') )
    CellIndexLayer.append(  FindCellIndex(layer4_x, layer4_y, 'layer4') )
    CellIndexLayer.append(  FindCellIndex(layer5_x, layer5_y, 'layer5') )
    CellIndexLayer.append(  FindCellIndex(layer6_x, layer6_y, 'layer6') )

    return np.array(CellIndexLayer)


def FindCellIndex(tr_x, tr_y, layer_name) : 
    
    layer_struct = {'layer1' : 32, 'layer2' : 64, 'layer3' : 32, 'layer4' : 16, 'layer5' : 16, 'layer6' : 8}

    
    # out_image = np.zeros( [layer_struct[layer_name], layer_struct[layer_name]] )
    
    x_idx = int(  (tr_x + 125)/250 *  layer_struct[layer_name] )
    y_idx = int(  (tr_y + 125)/250 * layer_struct[layer_name]  )
    
    return np.array([x_idx, y_idx])

def Vol_Cells():
    
    X0_ECAL = 3.897
    X0_HCAL = 2.357 
        
    Lambda_int_ECAL = 37.578
    Lambda_int_HCAL = 17.438
    
    layers=[32,64,32,16,16,8]
    X0_rad_t=[250/X0_ECAL/32, 250/X0_ECAL/64, 250/X0_ECAL/32, 250/X0_HCAL/16, 250/X0_HCAL/16, 250/X0_HCAL/8]
    X0_depth=[3, 16, 6, 1.5*Lambda_int_HCAL/X0_HCAL, 4.1*Lambda_int_HCAL/X0_HCAL, 1.8*Lambda_int_HCAL/X0_HCAL]
    
    V_X0=[]
    for i in range(6):
        V_X0.append(X0_rad_t[i]*X0_rad_t[i]*X0_depth[i])
        
        
    return V_X0
    

def En_Topo(gi, fi, types, ev):
    En = 0
    EnL = 0
    EnTopo = []
    for i in range(1,7):
        Topo = gi['TopoClusters'+str(i)][ev][0]
        Calo = fi['RealRes_'+str(types)+'Energy_Layer'+str(i)][ev][0]
            
        if(i==1 or i==3):
            size=32
        elif(i==4 or i ==5):
            size=16
        elif(i==2): 
            size=64
        elif(i==6):
            size = 8    
            
        TopoA=np.array(Topo[0:size,0:size])
        CaloA=np.array(Calo[0:size,0:size])

        for X in range(len(TopoA)):
            for Y in range(len(TopoA)): 
                TopoI=int(TopoA[X][Y])
                if TopoI!=0:
                    #print(X, Y, "En=",Calo[X][Y],'Topo N =', TopoI, 'layer =', i)
                    if len(EnTopo)<TopoI: EnTopo.append(Calo[X][Y])
                    else: EnTopo[TopoI-1]=EnTopo[TopoI-1]+CaloA[X][Y]
                        
    return EnTopo

def En_Topo_N(gi, fi, types, ev):
    En = 0
    EnL = 0
    EnTopo = []
    for i in range(1,7):
        Topo = gi['TopoClusters'+str(i)][ev][0]
        Calo = fi[str(types)+'_'+str(i)][ev]
            
        if(i==1 or i==3):
            size=32
        elif(i==4 or i ==5):
            size=16
        elif(i==2): 
            size=64
        elif(i==6):
            size = 8    
            
        TopoA=np.array(Topo[0:size,0:size])
        CaloA=np.array(Calo[0:size,0:size])

        for X in range(len(TopoA)):
            for Y in range(len(TopoA)): 
                TopoI=int(TopoA[X][Y])
                if TopoI!=0:
                    #print(X, Y, "En=",Calo[X][Y],'Topo N =', TopoI, 'layer =', i)
                    if len(EnTopo)<TopoI: EnTopo.append(Calo[X][Y])
                    else: EnTopo[TopoI-1]=EnTopo[TopoI-1]+CaloA[X][Y]
                        
    return EnTopo

In [6]:
#f_ch["Charged_4"][18]
#f["RealRes_ChargedEnergy_Layer4"][18]

In [7]:
layers=[32,64,32,16,16,8]

X0_ECAL = 3.897
X0_HCAL = 2.357 

Lambda_int_ECAL = 37.578
Lambda_int_HCAL = 17.438

depth=[0, 3*X0_ECAL/Lambda_int_ECAL,16*X0_ECAL/Lambda_int_ECAL,6*X0_ECAL/Lambda_int_ECAL, 1.5, 4.1,1.8 ]

LHED=[]
for ev in range(1000):
    if ev%100==0: print("ev",ev)
    Vol_Cell=Vol_Cells()
    Dens_En=[]
    Dens_En.append(0)
    C1=f["Trk_Theta"]    
    C2=f["Trk_Phi"]
    C3=f["Trk_X_indx"]   
    C4=f["Trk_Y_indx"]
    Tr=MakeTruthTrajectory(C1[ev], C2[ev], C3[ev], C4[ev])     
    
    for lay in range(6):
        #Lay_Ev=f["RealRes_ChargedEnergy_Layer"+str(lay+1)][ev][0]
        Lay_Ev=f_ch["Charged_"+str(lay+1)][ev]
        
        #CCWN=Cluster_Ch["TopoClusters"+str(lay+1)][ev][0][:layers[lay],:layers[lay]]
        CCN =Cluster_Charged["TopoClusters"+str(lay+1)][ev][0][:layers[lay],:layers[lay]]
        En_Lay=0
        for X in range(layers[lay]):
            for Y in range(layers[lay]):
                if int(CCN[X][Y])!=0:
                    ga_we=math.exp(-((Tr[lay][0]-X)**2+(Tr[lay][1]-Y)**2)/(0.00236672*layers[lay]**2))
                #print(X,Tr[lay][0],Y, Tr[lay][1],layers[lay], ga_we)
                    En_Lay+=Lay_Ev[X][Y]/Vol_Cell[lay]*ga_we
        
        Dens_En.append(En_Lay)

    #print("Rho",Dens_En)
    Delta_Dens=[]
    for i in range(1,7):
        Delta_Dens.append((Dens_En[i]-Dens_En[i-1])/(depth[i]-depth[i-1]))
    #print("D",depth)
    #print("DRho",Delta_Dens)

    ind=Delta_Dens.index(max(Delta_Dens))
    #print(ind)
    if Dens_En[ind+1]<Dens_En[ind]: ind-=1
    #print(ind)
    LHED.append(ind)



ev 0
ev 100
ev 200
ev 300
ev 400
ev 500
ev 600
ev 700
ev 800
ev 900


In [8]:
print(LHED)
hist=ROOT.TH1D("LHED","LHED",6,0,6)
for i in range(1000):
    hist.Fill(LHED[i])
hist.GetXaxis().SetTitle("LHED")
can=ROOT.TCanvas( "LHED","LHED",1600,900);
hist.Draw("")
can.Print("LHED1.png")    


[1, 4, 1, 1, 4, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 4, 1, 1, 1, 3, 1, 1, 1, 1, 1, 0, 3, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 3, 1, 0, 1, 0, 1, 0, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 3, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 3, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 4, 1, 1, 0, 1, 1, 0, 1, 1, 1, 3, 3, 1, 0, 1, 1, 1, 0, 1, 1, 1, 3, 1, 1, 3, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 4, 1, 0, 4, 1, 0, 0, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 4, 1, 3, 1, 1, 1, 1, 3, 1, 3, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 3, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 3, 1, 1, 0, 4, 1, 1, 1, 1, 1, 1, 0, 0, 4, 1, 0, 0, 1, 1, 0, 4, 1, 0, 0, 1, 0, 1, 1, 1, 3, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 3, 4, 1, 1, 1, 1, 4, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 4, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 4, 0, 0, 1, 1, 1, 1, 3, 0, 0, 1, 

Info in <TCanvas::Print>: png file LHED1.png has been created


In [9]:
def P_Tr_Comp(fi):
    Pt=[]
    for i in range(len(fi["Smeared_Track_Energy"])):
        En=fi["Smeared_Track_Energy"][i]

        ptr=math.sqrt(En*En-139.57018*139.57018)
        Pt.append(ptr)
    return Pt



P_Track=P_Tr_Comp(f)

In [10]:
N_Topo_Charged=[]
layers=[32,64,32,16,16,8]


for ev in range(len(f_ch["Charged_1"])):
    if ev%100==0: print("Yo ev: "+str(ev))
    
    #define layer of first interaction
    lay=LHED[ev]
    #print(lay)
    
    #Lay_Ev=f["RealRes_ChargedEnergy_Layer"+str(lay+1)][ev][0]
    Lay_Ev=f_ch["Charged_"+str(lay+1)][ev]
        
    #CCWN=Cluster_Ch["TopoClusters"+str(lay+1)][ev][0][:layers[lay],:layers[lay]]
    Topo_Ev =Cluster_Charged["TopoClusters"+str(lay+1)][ev][0][:layers[lay],:layers[lay]]
    
    
  
    AverPos=[]
    E_Topo_Lay=[]
    for X in range(layers[lay]):
        for Y in range(layers[lay]):
            TopoInt=int(Topo_Ev[X][Y])
            if TopoInt!=0:
                    
                    #find Center Of Topocluster (AverPos) and
                    # TotalEnergy (E_Topo) within that layer such <X>=Ei Xi / Etot
                if len(AverPos)<TopoInt: 
                    AverPos.append([Lay_Ev[X][Y]*X ,  Lay_Ev[X][Y]*Y])
                    E_Topo_Lay.append(Lay_Ev[X][Y])
                else: 
                    AverPos[TopoInt-1][0]=AverPos[TopoInt-1][0]+Lay_Ev[X][Y]*X
                    AverPos[TopoInt-1][1]=AverPos[TopoInt-1][1]+Lay_Ev[X][Y]*Y
                    E_Topo_Lay[TopoInt-1]=E_Topo_Lay[TopoInt-1]+Lay_Ev[X][Y]

    #print(AverPos, ev)
    if not AverPos:
        
        N_Topo_Charged.append(-10)
        #print("Skipping ev: ", ev)
        continue
        
    #print(AverPos)
    #print(E_Topo_Lay)
    for j,r in enumerate(AverPos):
        for i in range(2):
            r[i]=int(r[i]/E_Topo_Lay[j]+0.5)

        # AverPos Contains mean position of each topocluster
    #print("Center for each clust.: ",AverPos, "", lay+1)
        
        
        #defining track position
    C1=f["Trk_Theta"]    
    C2=f["Trk_Phi"]
    C3=f["Trk_X_indx"]   
    C4=f["Trk_Y_indx"]
    Traj_Mark=MakeTruthTrajectory(C1[ev], C2[ev], C3[ev], C4[ev])        
    #print("Real Position of Track: ",Traj_Mark[lay])

        #compute distances Clus-Track
    D=[]
    for r in AverPos:
        D.append(math.sqrt((r[0]-Traj_Mark[lay][0])**2+(r[1]-Traj_Mark[lay][1])**2))
    #print("Distances Clus-Track: ",D)
        
        #return which clust is nearest
    CN=D.index(min(D))
    #print("Cluster nearest to track: ",CN)
        
        #fill a list for each event with number of cluster nearest to track
    N_Topo_Charged.append(CN)



Yo ev: 0
Yo ev: 100
Yo ev: 200
Yo ev: 300
Yo ev: 400
Yo ev: 500
Yo ev: 600
Yo ev: 700
Yo ev: 800
Yo ev: 900


In [11]:
for i,N in enumerate(N_Topo_Charged):
    if N==-10 and i!=108: 
        print(i,N) 
        break     

168 -10


In [12]:
print(LHED[168])

0


In [13]:
E_Pred=[]
for ev in range(1000):
    Total_E_Topo=En_Topo_N(Cluster_Charged,f_ch,"Charged",ev)
    #print(Total_E_Topo[N_CL[ev]])
    if N_Topo_Charged[ev]==-10:
            E_Pred.append(-10)
            continue
            
    E_Pred.append(Total_E_Topo[N_Topo_Charged[ev]]/P_Track[ev])


In [14]:
#print(E_Pred)

In [15]:
#hist=ROOT.TH1D("LHED","LHED",6,0,6)
#k=0
#for i in range(1000):
#    if N_Topo_Charged[i]==-10:
#        continue
#    k+=1
#    hist.Fill(LHED[i])
#hist.GetXaxis().SetTitle("LHED")
#can=ROOT.TCanvas( "LHED","LHED",1600,900);
#hist.Draw("")
#can.Print("LHED.png")    

In [16]:
h_Epred= ROOT.TH3D("h_Epred","h_Epred",300,0.,40e3,6,0,6,50,0.,1.)
h_Epred_prof= ROOT.TProfile2D("h_Epred_prof","h_Epred_prof",300,0.,40e3,6,0,6,0.,1.)

In [17]:
k=0
for ev in range(1000):
    if N_Topo_Charged[ev]==-10: 
        continue
    k+=1    
    h_Epred.Fill(P_Track[ev], LHED[ev], E_Pred[ev])
    h_Epred_prof.Fill(P_Track[ev], LHED[ev], E_Pred[ev])
    
    
print(k)

986


In [21]:
#h_EP=ROOT.TH1D("1bin","1bin",100,0,1)
E_MED=0
for ev in range(1000):
    #print(E_Pred[ev])
    if E_Pred[ev]!=-10:
        E_MED+=E_Pred[ev]/1000

print(E_MED)



0.6324962993256177


In [18]:
f_o=ROOT.TFile("Epred.root","recreate")
h_Epred.Write()
h_Epred_prof.Write()
f_o.Close()

In [23]:
N_Topo_Total=[]
layers=[32,64,32,16,16,8]


for ev in range(len(f_ch["Charged_1"])):
    if ev%100==0: print("Yo ev: "+str(ev))
    
        #define layer of first interaction
    lay=LHED[ev]
    #print(lay)
    
    Lay_Ev=f["RealRes_TotalEnergy_Layer"+str(lay+1)][ev][0][:layers[lay],:layers[lay]]
    Topo_Ev=Cluster_Total["TopoClusters"+str(lay+1)][ev][0][:layers[lay],:layers[lay]]
    
  
    AverPos=[]
    E_Topo_Lay=[]
    for X in range(layers[lay]):
        for Y in range(layers[lay]):
            TopoInt=int(Topo_Ev[X][Y])
            if TopoInt!=0:
                    
                    #find Center Of Topocluster (AverPos) and
                    # TotalEnergy (E_Topo) within that layer such <X>=Ei Xi / Etot
                if len(AverPos)<TopoInt: 
                    AverPos.append([Lay_Ev[X][Y]*X ,  Lay_Ev[X][Y]*Y])
                    E_Topo_Lay.append(Lay_Ev[X][Y])
                else: 
                    AverPos[TopoInt-1][0]=AverPos[TopoInt-1][0]+Lay_Ev[X][Y]*X
                    AverPos[TopoInt-1][1]=AverPos[TopoInt-1][1]+Lay_Ev[X][Y]*Y
                    E_Topo_Lay[TopoInt-1]=E_Topo_Lay[TopoInt-1]+Lay_Ev[X][Y]

    if not AverPos: 
        N_Topo_Total.append(-10)
        #print("Skipping ev: ", ev)
        continue
        
    #print(AverPos)
    #print(E_Topo_Lay)
    for j,r in enumerate(AverPos):
        for i in range(2):
            r[i]=int(r[i]/E_Topo_Lay[j]+0.5)

        # AverPos Contains mean position of each topocluster
    #print("Center for each clust.: ",AverPos, "", lay+1)
        
        
        #defining track position
    C1=f["Trk_Theta"]    
    C2=f["Trk_Phi"]
    C3=f["Trk_X_indx"]   
    C4=f["Trk_Y_indx"]
    Traj_Mark=MakeTruthTrajectory(C1[ev], C2[ev], C3[ev], C4[ev])        
    #print("Real Position of Track: ",Traj_Mark[lay])

        #compute distances Clus-Track
    D=[]
    for r in AverPos:
        D.append(math.sqrt((r[0]-Traj_Mark[lay][0])**2+(r[1]-Traj_Mark[lay][1])**2))
    #print("Distances Clus-Track: ",D)
        
        #return which clust is nearest
    CN=D.index(min(D))
    #print("Cluster nearest to track: ",CN)
        
        #fill a list for each event with number of cluster nearest to track
    N_Topo_Total.append(CN)


    
    



Yo ev: 0
Yo ev: 100
Yo ev: 200
Yo ev: 300
Yo ev: 400
Yo ev: 500
Yo ev: 600
Yo ev: 700
Yo ev: 800
Yo ev: 900


In [36]:
NTT=N_Topo_Total
NTC=N_Topo_Charged
print(len(NTC))

1000


In [41]:
Distr=[]
h_EN= ROOT.TH1D("(Ech - Epred)/En","(Ech - Epred)/En",100,-2,2)
k=0
for ev in range(len(f["Smeared_Track_Energy"])):
    
    #Neutral part on total topocluster
    E_N=En_Topo(Cluster_Total,f,"Neutral",ev)
    
    #total part in total topocluster
    E_Tot=En_Topo(Cluster_Total,f,"Total",ev)
    
    #E_Ch=En_Topo_N(Cluster_Charged,f_ch,"Charged",ev)
    
    #charged energy predicted 
    #EPR=h_Epred_prof.GetBinContent(h_Epred_prof.GetXaxis().FindBin(P_Track[ev]),h_Epred_prof.GetYaxis().FindBin(LHED[ev]))
    #E_Pred=EPR*P_Track[ev]
    
    #print(E_Pred)
    if NTT[ev]==-10:
        continue
    #print(EPR,E_Ch[NTC[ev]]/P_Track[ev])
    k+=1
    #print(NTT[ev])
    h_EN.Fill(E_MED - (E_Tot[NTT[ev]]-E_N[NTT[ev]])/P_Track[ev])

print(k)

994


Warning in <TROOT::Append>: Replacing existing TH1: (Ech - Epred)/En (Potential memory leak).


In [42]:
h_EN.GetXaxis().SetTitle("(Ech - Epred)/En")
can=ROOT.TCanvas( "(Ech - Epred)/ En","(Ech - Epred)/ En",1600,900);
h_EN.Draw("")
can.Print("DE_ch_3.png")

Info in <TCanvas::Print>: png file DE_ch_3.png has been created


In [ ]:
f_o=ROOT.TFile("Epred.root","recreate")
h_EN.Write()
h_Epred.Write()
h_Epred_prof.Write()
#hRes.Write()                                                                                                                                                                                               
#hEnergy.Write()                                                                                                                                                                                            
f_o.Close()


In [ ]:
h_EN2= ROOT.TH1D("(En - ENN)/En","(En - ENN)/En",100,-2,2)
k=0
for ev in range(1000):
    
    #Neutral part on total topocluster
    E_N=En_Topo(Cluster_Total,f,"Neutral",ev)
    
    #total part in total topocluster
    E_Tot=En_Topo(Cluster_Total,f,"Total",ev)
    
    #E_Ch=En_Topo_N(Cluster_Charged,f_ch,"Charged",ev)
    
    #charged energy predicted 
    #EPR=h_Epred_prof.GetBinContent(h_Epred_prof.GetXaxis().FindBin(P_Track[ev]),h_Epred_prof.GetYaxis().FindBin(LHED[ev]))
    #E_Pred=EPR*P_Track[ev]
    
    #print(E_Pred)
    if NTT[ev]==-10:
        continue
    #print(EPR,E_Ch[NTC[ev]]/P_Track[ev])
    k+=1
    #print(NTT[ev])
    h_EN.Fill(E_MED - (E_Tot[NTT[ev]]-E_N[NTT[ev]])/P_Track[ev])

print(k)